In [ ]:
import numpy as np
import pandas as pd
from scipy.stats import poisson

MAX_GOLES = 5
RANGO_GOLES = np.arange(MAX_GOLES + 1)
RESULTADOS = [(i, j) for i in RANGO_GOLES for j in RANGO_GOLES]


def signo_desenlace(goles_local: int, goles_visitante: int) -> int:
    """
    Devuelve el desenlace (1X2) de un marcador:
     1 -> Victoria local (i > j)
     0 -> Empate (i == j)
    -1 -> Victoria visitante (i < j)
    """
    if goles_local > goles_visitante:
        return 1
    elif goles_local < goles_visitante:
        return -1
    return 0


def calcular_probabilidades_partido(lambda_local: float, lambda_visitante: float) -> np.ndarray:
    """
    Función 1: Calcula el vector de 36 probabilidades conjuntas P(G_L = i, G_V = j)
    asumiendo independencia entre dos distribuciones de Poisson.
    El índice k del vector corresponde al resultado (i, j) con k = 6*i + j.
    """
    p_local = poisson.pmf(RANGO_GOLES, lambda_local)
    p_visitante = poisson.pmf(RANGO_GOLES, lambda_visitante)

    matriz_prob = np.outer(p_local, p_visitante)
    return matriz_prob.ravel()



def construir_matriz_utilidad(
    resultados: list[tuple[int, int]],
    pts_exacto: float = 2.0,
    pts_tendencia: float = 1.0
) -> np.ndarray:
    """
    Construye la matriz de utilidad U de dimensión nxn.
    Filas (s): resultado real ocurrido.
    Columnas (s_hat): resultado exacto predicho (decisión).

    Parámetros:
    - resultados: lista de tuplas con los resultados posibles.
    - pts_exacto: puntos asignados si se acierta el resultado exacto (ej. 2.0).
    - pts_tendencia: puntos asignados si se acierta el desenlace (1X2) (ej. 1.0).
    """
    n = len(resultados)
    U = np.zeros((n, n), dtype=float)

    for idx_real, (i, j) in enumerate(resultados):
        desenlace_real = signo_desenlace(i, j)
        for idx_pred, (i_hat, j_hat) in enumerate(resultados):
            desenlace_pred = signo_desenlace(i_hat, j_hat)

            if i == i_hat and j == j_hat:
                U[idx_real, idx_pred] = float(pts_exacto)  # Acierto exacto
            elif desenlace_real == desenlace_pred:
                U[idx_real, idx_pred] = float(pts_tendencia)  # Acierto de desenlace (1X2)
            else:
                U[idx_real, idx_pred] = 0.0  # Desacierto total
    return U


def calcular_valor_esperado(probabilidades: np.ndarray, matriz_utilidad: np.ndarray) -> np.ndarray:
    """
    Calcula el valor esperado de puntos para cada una de las 36 decisiones posibles.
    VE(s_hat) = sum_s P(s) * U(s, s_hat)
    """
    # Producto punto entre el vector de probabilidades (1x36) y la matriz de utilidad (36x36)
    return probabilidades @ matriz_utilidad




In [ ]:
partidos = [
    {"local": "Equipo Rojo",     "visitante": "Equipo Azul",     "lam_l": 0.5, "lam_v": 1.0},
    {"local": "Equipo Amarillo", "visitante": "Equipo Rojo",     "lam_l": 1.5, "lam_v": 0.5},
    {"local": "Equipo Verde",    "visitante": "Equipo Azul",     "lam_l": 2.0, "lam_v": 1.0},
    {"local": "Equipo Amarillo", "visitante": "Equipo Verde",    "lam_l": 1.5, "lam_v": 2.0},
]

U = construir_matriz_utilidad(RESULTADOS)
resumen = []

for p in partidos:
    probs = calcular_probabilidades_partido(p["lam_l"], p["lam_v"])
    ve = calcular_valor_esperado(probs, U)

    max_ve = np.max(ve)
    indices_optimos = np.where(np.isclose(ve, max_ve, atol=1e-12))[0]
    predicciones_optimas = [f"{RESULTADOS[idx][0]}-{RESULTADOS[idx][1]}" for idx in indices_optimos]

    resumen.append({
        "Partido": f"{p['local']} vs. {p['visitante']}",
        "Lambda local": p["lam_l"],
        "Lambda visitante": p["lam_v"],
        "Suma P(s) (0..5)": round(np.sum(probs), 4),
        "Predicción óptima": ", ".join(predicciones_optimas),
        "Valor esperado": round(max_ve, 4),
    })

df_resumen = pd.DataFrame(resumen)
print(df_resumen.to_string(index=False))

                         Partido  Lambda local  Lambda visitante  Suma P(s) (0..5) Predicción óptima  Valor esperado
     Equipo Rojo vs. Equipo Azul           0.5               1.0            0.9994               0-1          0.6924
 Equipo Amarillo vs. Equipo Rojo           1.5               0.5            0.9955               1-0          0.8192
    Equipo Verde vs. Equipo Azul           2.0               1.0            0.9829     1-0, 2-0, 2-1          0.6887
Equipo Amarillo vs. Equipo Verde           1.5               2.0            0.9791               1-2          0.5676


In [ ]:
import numpy as np

partido = {
    "local": "Equipo Rojo",
    "visitante": "Equipo Azul",
    "lam_l": 0.5,
    "lam_v": 1.0
}

U = construir_matriz_utilidad(RESULTADOS)
probs = calcular_probabilidades_partido(partido["lam_l"], partido["lam_v"])
ve = calcular_valor_esperado(probs, U)

max_ve = np.max(ve)
indices_optimos = np.where(np.isclose(ve, max_ve, atol=1e-12))[0]
predicciones_optimas = [f"{RESULTADOS[idx][0]}-{RESULTADOS[idx][1]}" for idx in indices_optimos]

print(f"Partido:           {partido['local']} vs. {partido['visitante']}")
print(f"Lambda local:      {partido['lam_l']}")
print(f"Lambda visitante:  {partido['lam_v']}")
print(f"Suma P(s) (0..5):  {round(np.sum(probs), 4)}")
print(f"Predicción óptima: {', '.join(predicciones_optimas)}")
print(f"Valor esperado:    {round(max_ve, 4)}")

Partido:           Equipo Rojo vs. Equipo Azul
Lambda local:      0.5
Lambda visitante:  1.0
Suma P(s) (0..5):  0.9994
Predicción óptima: 0-1
Valor esperado:    0.6924


In [ ]:
import numpy as np

MAX_GOLES = 8
RANGO_GOLES = np.arange(MAX_GOLES + 1)
RESULTADOS = [(i, j) for i in RANGO_GOLES for j in RANGO_GOLES]


partido = {
    "local": "Equipo Rojo",
    "visitante": "Equipo Azul",
    "lam_l": 0.5,
    "lam_v": 1.0
}

U = construir_matriz_utilidad(RESULTADOS,4,2)
probs = calcular_probabilidades_partido(partido["lam_l"], partido["lam_v"])
ve = calcular_valor_esperado(probs, U)

max_ve = np.max(ve)
indices_optimos = np.where(np.isclose(ve, max_ve, atol=1e-12))[0]
predicciones_optimas = [f"{RESULTADOS[idx][0]}-{RESULTADOS[idx][1]}" for idx in indices_optimos]

print(f"Partido:           {partido['local']} vs. {partido['visitante']}")
print(f"Lambda local:      {partido['lam_l']}")
print(f"Lambda visitante:  {partido['lam_v']}")
print(f"Suma P(s) (0..5):  {round(np.sum(probs), 4)}")
print(f"Predicción óptima: {', '.join(predicciones_optimas)}")
print(f"Valor esperado:    {round(max_ve, 4)}")

Partido:           Equipo Rojo vs. Equipo Azul
Lambda local:      0.5
Lambda visitante:  1.0
Suma P(s) (0..5):  1.0
Predicción óptima: 0-1
Valor esperado:    1.386
